# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [1]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/OHF-Voice/micro-wake-word microWakeWord
!pip install -e ./microWakeWord

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-svfaztpu
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-svfaztpu
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!rm -rf piper-sample-generator

In [7]:
!rm -rf piper-sample-generator
!git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 135 (delta 53), reused 44 (delta 44), pack-reused 66 (from 1)
Receiving objects: 100% (135/135), 1.03 MiB | 3.32 MiB/s, done.
Resolving deltas: 100% (61/61), done.
--2026-07-08 13:58:00--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-08T14%3A50%3A12Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-4

In [2]:
target_word = 'Paro'

import os
import sys

!rm -rf piper-sample-generator
!git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install torch torchaudio piper-phonemize-cross==1.2.1
!sed -i 's/torch.load(model_path)/torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir generated_samples

Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 135 (delta 53), reused 44 (delta 44), pack-reused 66 (from 1)
Receiving objects: 100% (135/135), 1.03 MiB | 15.25 MiB/s, done.
Resolving deltas: 100% (61/61), done.
--2026-07-08 16:57:24--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-08T17%3A31%3A41Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-

In [9]:
!sed -i 's/torch.load(model_path)/torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py

In [3]:
# Generates a larger amount of wake word samples.
# Start here when trying to improve your model.
# See https://github.com/rhasspy/piper-sample-generator for the full set of
# parameters. In particular, experiment with noise-scales and noise-scale-ws,
# generating negative samples similar to the wake word, and generating many more
# wake word samples, possibly with different phonetic pronunciations.

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1000 \
--batch-size 100 \
--output-dir generated_samples

DEBUG:__main__:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/10 complete
DEBUG:__main__:Batch 2/10 complete
DEBUG:__main__:Batch 3/10 complete
DEBUG:__main__:Batch 4/10 complete
DEBUG:__main__:Batch 5/10 complete
DEBUG:__main__:Batch 6/10 complete
DEBUG:__main__:Batch 7/10 complete
DEBUG:__main__:Batch 8/10 complete
DEBUG:__main__:Batch 9/10 complete
DEBUG:__main__:Batch 10/10 complete
INFO:__main__:Done


In [4]:
from google.colab import files
uploaded = files.upload()  # wakeword_samples.zip を選択
!unzip -o wakeword_samples.zip -d generated_samples/
!mv generated_samples/wakeword_samples/*.wav generated_samples/
!rmdir generated_samples/wakeword_samples
!ls generated_samples/*.wav | wc -l

Saving wakeword_samples.zip to wakeword_samples.zip
Archive:  wakeword_samples.zip
   creating: generated_samples/wakeword_samples/
  inflating: generated_samples/wakeword_samples/paro_grandma_r220_pm100.wav  
  inflating: generated_samples/wakeword_samples/paro_rocko_r220_pp0.wav  
  inflating: generated_samples/wakeword_samples/paro_reed_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandpa_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandpa_r160_pp0.wav  
  inflating: generated_samples/wakeword_samples/paro_sandy_r220_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_eddy_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandpa_r220_pp0.wav  
  inflating: generated_samples/wakeword_samples/paro_flo_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandma_r160_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_sandy_r160_pm100.wav  
  inflating: generated_samples/w

In [13]:
!unzip -o wakeword_samples.zip -d generated_samples/
!mv generated_samples/wakeword_samples/*.wav generated_samples/
!rmdir generated_samples/wakeword_samples
!ls generated_samples/*.wav | wc -l

Archive:  wakeword_samples.zip
   creating: generated_samples/wakeword_samples/
  inflating: generated_samples/wakeword_samples/paro_grandma_r220_pm100.wav  
  inflating: generated_samples/wakeword_samples/paro_rocko_r220_pp0.wav  
  inflating: generated_samples/wakeword_samples/paro_reed_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandpa_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandpa_r160_pp0.wav  
  inflating: generated_samples/wakeword_samples/paro_sandy_r220_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_eddy_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandpa_r220_pp0.wav  
  inflating: generated_samples/wakeword_samples/paro_flo_r130_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_grandma_r160_pp100.wav  
  inflating: generated_samples/wakeword_samples/paro_sandy_r160_pm100.wav  
  inflating: generated_samples/wakeword_samples/paro_shelley_r130_pm100.wav  
  infl

In [15]:
sample_row = next(iter(rir_dataset))
print(sample_row.keys())
print(type(sample_row['audio']))

dict_keys(['audio'])
<class 'datasets.features._torchcodec.AudioDecoder'>


In [22]:
!rm -rf fma_16k

In [8]:
import os
for d in ["mit_rirs", "audioset_16k", "fma", "fma_16k"]:
    print(d, os.path.exists(d))

mit_rirs True
audioset_16k True
fma True
fma_16k True


In [9]:
!rm -rf mit_rirs audioset_16k fma fma_16k

In [10]:
import datasets
import scipy
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

def decoder_to_mono_int16(row_audio):
    samples = row_audio.get_all_samples()
    audio_array = samples.data.numpy()
    if audio_array.ndim == 2:
        audio_array = audio_array.mean(axis=0)
    audio_array = np.clip(audio_array, -1.0, 1.0)
    return (audio_array * 32767).astype(np.int16), samples.sample_rate

print("=== RIR ===")
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for i, row in enumerate(tqdm(rir_dataset)):
        pcm, rate = decoder_to_mono_int16(row['audio'])
        scipy.io.wavfile.write(os.path.join(output_dir, f"rir_{i}.wav"), rate, pcm)
    print("RIR done")
else:
    print("RIR skipped (folder exists)")

print("=== Audioset ===")
output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    audioset_dataset = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
    MAX_AUDIOSET_CLIPS = 2000
    for i, row in enumerate(tqdm(audioset_dataset, total=MAX_AUDIOSET_CLIPS)):
        if i >= MAX_AUDIOSET_CLIPS:
            break
        pcm, rate = decoder_to_mono_int16(row['audio'])
        scipy.io.wavfile.write(os.path.join(output_dir, f"audioset_{i}.wav"), rate, pcm)
    print("Audioset done")
else:
    print("Audioset skipped (folder exists)")

print("=== FMA ===")
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    get_ipython().system('wget -O {out_dir} {link}')
    get_ipython().system('cd {output_dir} && unzip -q {fname}')

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)
    fma_files = [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]
    fma_dataset = datasets.Dataset.from_dict({"audio": fma_files})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for i, row in enumerate(tqdm(fma_dataset)):
        pcm, rate = decoder_to_mono_int16(row['audio'])
        name = Path(fma_files[i]).stem + ".wav"
        scipy.io.wavfile.write(os.path.join(output_dir, name), rate, pcm)
    print("FMA done")
else:
    print("FMA skipped (folder exists)")

print("=== ALL DONE ===")

=== RIR ===


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [00:39,  6.87it/s]


RIR done
=== Audioset ===


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [02:51<00:00, 11.68it/s]


Audioset done
=== FMA ===
--2026-07-08 17:11:46--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 3.166.152.110, 3.166.152.65, 3.166.152.105, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.110|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?response-content-type=application%2Fzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1783534306&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9YXBwbGljYXRpb24lMkZ6aXAmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1p

100%|██████████| 210/210 [00:15<00:00, 13.85it/s]

FMA done
=== ALL DONE ===


In [35]:
print("START audioset")

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    print("audioset: starting streaming download")
    os.mkdir(output_dir)

    audioset_dataset = datasets.load_dataset(
        "agkphysics/AudioSet", "balanced", split="train", streaming=True
    )

    MAX_AUDIOSET_CLIPS = 2000  # 元のtar1本分(約2000件弱)相当を目安に取得
    count = 0
    for i, row in enumerate(tqdm(audioset_dataset, total=MAX_AUDIOSET_CLIPS)):
        if count >= MAX_AUDIOSET_CLIPS:
            break
        pcm, rate = decoder_to_mono_int16(row['audio'])
        name = f"audioset_{i}.wav"
        scipy.io.wavfile.write(os.path.join(output_dir, name), rate, pcm)
        count += 1
    print("audioset: done,", count, "files")
else:
    print("audioset: skipped, folder already exists")

print("END audioset")

START audioset
audioset: starting streaming download


README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [02:12<00:00, 15.05it/s]

audioset: done, 2000 files
END audioset


In [32]:
print("START fma")

output_dir = "./fma"
if not os.path.exists(output_dir):
    print("fma: starting download")
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    get_ipython().system('wget -O {out_dir} {link}')
    get_ipython().system('cd {output_dir} && unzip -q {fname}')

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    fma_files = [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]
    print("fma: mp3 files found:", len(fma_files))
    fma_dataset = datasets.Dataset.from_dict({"audio": fma_files})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    count = 0
    for i, row in enumerate(tqdm(fma_dataset)):
        pcm, rate = decoder_to_mono_int16(row['audio'])
        name = Path(fma_files[i]).stem + ".wav"
        scipy.io.wavfile.write(os.path.join(output_dir, name), rate, pcm)
        count += 1
    print("fma: done,", count, "files")
else:
    print("fma: skipped, folder already exists")

print("END fma")

START fma
fma: starting download
--2026-07-08 15:02:50--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.55, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&response-content-type=application%2Fzip&user_id=public&Expires=1783526570&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxl

100%|██████████| 210/210 [00:16<00:00, 12.79it/s]

fma: done, 210 files
END fma


In [11]:
import glob
print("RIR:", len(glob.glob("mit_rirs/*.wav")))
print("Audioset:", len(glob.glob("audioset_16k/*.wav")))
print("FMA:", len(glob.glob("fma_16k/*.wav")))

RIR: 270
Audioset: 2000
FMA: 210


In [40]:
import os
print(os.listdir("/content/microWakeWord"))

['README.md', '.gitignore', 'benchmarks', '.git', 'microwakeword.egg-info', 'documentation', 'microwakeword', 'pyproject.toml', 'notebooks', 'LICENSE', 'setup.py', 'etc', '.flake8']


In [41]:
import sys
print("/content/microWakeWord" in sys.path)
print([p for p in sys.path if "microWakeWord" in p or "microwakeword" in p])

False
[]


In [12]:
import sys
sys.path.insert(0, "/content/microWakeWord")

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )

In [43]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [14]:
from mmap_ninja.ragged import RaggedMmap

In [15]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [20]:
import os
for name in ["dinner_party", "dinner_party_eval", "no_speech", "speech"]:
    d = f"negative_datasets/{name}"
    z = f"negative_datasets/{name}.zip"
    print(name, "dir_exists:", os.path.exists(d), "zip_exists:", os.path.exists(z))

dinner_party dir_exists: True zip_exists: False
dinner_party_eval dir_exists: True zip_exists: False
no_speech dir_exists: True zip_exists: False
speech dir_exists: True zip_exists: False


In [21]:
!rm -rf negative_datasets

In [22]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = "./negative_datasets"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
filenames = ["dinner_party.zip", "dinner_party_eval.zip", "no_speech.zip", "speech.zip"]

for fname in filenames:
    zip_path = os.path.join(output_dir, fname)
    link = link_root + fname
    target_dir = os.path.join(output_dir, fname.replace(".zip", ""))

    if not os.path.exists(target_dir):
        get_ipython().system('wget -O "{zip_path}" "{link}"')
        get_ipython().system('unzip -q "{zip_path}" -d "{output_dir}"')
        os.remove(zip_path)

--2026-07-08 17:35:02--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 3.166.152.110, 3.166.152.44, 3.166.152.105, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.110|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dinner_party.zip%3B+filename%3D%22dinner_party.zip%22%3B&user_id=public&X-Xet-Cas-Uid=public&response-content-type=application%2Fzip&Expires=1783535703&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjLzIyOGQ3ZTcyY2Q1ZmRjNGU2ZTU3ZGEzNmI4OGE0YzIyN2QzNGNiOGRjNDQwNDEwNzhiNGM0YjY1ZGM3NTg0OGRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04JTI3JTI3ZGlubmVyX3Bh

In [23]:
import yaml

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/wakeword"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 32
config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)

print("saved")

saved


In [24]:
!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
2026-07-08 17:45:50.821337: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1783532750.822805   16145 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (32, 204, 40)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (32, 204, 1, 4

In [4]:
import os
print("trained_models exists:", os.path.exists("trained_models"))
print("cwd:", os.getcwd())
print(os.listdir("."))

trained_models exists: False
cwd: /content
['.config', 'sample_data']
